In [ ]:
from openai import OpenAI

# 初始化客户端（如果你用 vLLM 本地部署，记得换成对应 base_url）
client = OpenAI(base_url="http://localhost:22999/v1", api_key="EMPTY")


def multi_query_rewrite(query: str, n: int = 5, model: str = "gpt-4o-mini"):
    """
    输入 query，返回多个改写后的 query
    """
    prompt = f"""
你是一个查询改写助手。请根据用户的问题，生成 {n} 个不同但相关的检索查询，
保证覆盖更多的相关语义和表述方式。nvidia

用户问题: "{query}"
请直接输出改写后的查询，每个一行。
"""

    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7
    )

    rewrites = [
        line.strip("-• ").strip()
        for line in response.choices[0].message.content.split("\n")
        if line.strip()
    ]
    return rewrites[:n]

In [3]:
query = "人工智能对未来就业的影响"
rewrites = multi_query_rewrite(query, n=5, model="./Models/Qwen2.5-14B-Instruct")

print("原始 Query:", query)
print("\n改写后的 Queries:")
for i, q in enumerate(rewrites, 1):
    print(f"{i}. {q}")

原始 Query: 人工智能对未来就业的影响

改写后的 Queries:
1. 人工智能对就业的未来影响
2. 未来就业趋势与人工智能的关系
3. 人工智能如何改变未来的就业市场
4. 探讨人工智能对未来工作岗位的影响
5. 分析人工智能技术对就业市场的长远影响


In [ ]:
from openai import OpenAI
client = OpenAI(base_url="http://localhost:22999/v1", api_key="EMPTY")

def research_rewrite(question, n_variants=3):
    """
    研究版的 rewrite: 结合多路改写 + 问题拆解
    """
    prompt = f"""
你是一个信息检索专家。给定一个用户问题：
"{question}"

请你生成 {n_variants} 个不同的查询，每个查询必须满足以下约束：
1. 至少包含 1 个 **语义扩展**的改写（multi-query），即保持问题核心但换不同角度/不同表达。
2. 至少包含 1 个 **子问题拆解**（decomposition），即把复杂问题拆成具体可检索的子问题。
3. 至少包含 1 个 **对立/反向视角**，保证检索覆盖不同立场。

输出时用 JSON 格式返回，字段为：
[
  {{ "type": "multi-query", "query": "..." }},
  {{ "type": "decomposition", "query": "..." }},
  {{ "type": "decomposition", "query": "..." }},
  {{ "type": "multi-query", "query": "..." }},
  {{ "type": "contrastive", "query": "..." }}
]
    """
    response = client.chat.completions.create(
        model="./Models/Qwen2.5-14B-Instruct",  # 你本地跑的模型名，替换成合适的
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
        max_tokens=512
    )
    return response.choices[0].message.content


def rewrite_pipeline(questions, model="./Models/Qwen2.5-14B-Instruct"):
    """
    研究版 rewrite pipeline
    输入：问题列表
    输出：每个问题的多路改写（去重后）
    """
    all_results = {}
    for q in questions:
        rewrites = research_rewrite(q)
        all_results[q] = rewrites
    return all_results

In [17]:
questions = [
    "烧水"
]

results = rewrite_pipeline(questions)

import pprint
pprint.pprint(results)


{'烧水': '```json\n'
       '[\n'
       '  {\n'
       '    "type": "multi-query",\n'
       '    "query": "如何快速煮沸一壶水"\n'
       '  },\n'
       '  {\n'
       '    "type": "decomposition",\n'
       '    "query": "使用电热水壶烧水需要多长时间"\n'
       '  },\n'
       '  {\n'
       '    "type": "decomposition",\n'
       '    "query": "在没有电力的情况下如何烧开一壶水"\n'
       '  },\n'
       '  {\n'
       '    "type": "multi-query",\n'
       '    "query": "怎样有效地节约能源来烧水"\n'
       '  },\n'
       '  {\n'
       '    "type": "contrastive",\n'
       '    "query": "为什么有些人认为煮沸水是不必要的"\n'
       '  }\n'
       ']\n'
       '```'}


__________Need Test_________

In [ ]:
from openai import OpenAI
client = OpenAI(base_url="http://localhost:22999/v1", api_key="EMPTY")
import textwrap

def research_rewrite(question, n_variants=3):
    """
    研究版的 rewrite: 结合多路改写 + 问题拆解
    """
    prompt = textwrap.dedent(f"""
            你是一个信息检索专家。给定一个用户问题：
            "{question}"

            请你生成 {n_variants} 个不同的查询，每个查询必须满足以下约束：
            1. 至少包含 1 个 **语义扩展**的改写（multi-query），即保持问题核心但换不同角度/不同表达。
            2. 至少包含 1 个 **子问题拆解**（decomposition），即把复杂问题拆成具体可检索的子问题。
            3. 至少包含 1 个 **对立/反向视角**，保证检索覆盖不同立场。

            要求：
            - 每个改写独立占一行
            - 不要加序号、符号或额外解释
            - 保持自然语言形式
            - 输出示例：
            查询1
            查询2
            查询3
            ...
        """)
    response = client.chat.completions.create(
        model="./Models/Qwen2.5-14B-Instruct",  # 你本地跑的模型名，替换成合适的
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
        max_tokens=512
    )
    return response.choices[0].message.content


def rewrite_pipeline(questions, model="./Models/Qwen2.5-14B-Instruct"):
    """
    研究版 rewrite pipeline
    输入：问题列表
    输出：每个问题的多路改写（去重后）
    """
    all_results = {}
    for q in questions:
        rewrites = research_rewrite(q)
        all_results[q] = rewrites
    return all_results

In [ ]:
prompt = f"""
你是一个检索查询改写助手。
请根据下面的用户问题，生成 {n} 个不同但语义相关的检索查询。
要求：
- 每个改写独立占一行
- 不要加序号、符号或额外解释
- 保持自然语言形式
- 输出示例：
查询1
查询2
查询3
...

用户问题: "{query}"
"""

In [3]:
import re
import textwrap
from openai import OpenAI

class QueryRewriter:
    """
    QueryRewriter: 一个可插入 RAG pipeline 的查询改写组件。
    支持 multi-query、decomposition、opposite-view 改写策略。
    """

    def __init__(self, model: str = "gpt-4o-mini", base_url: str = "http://localhost:22999/v1", api_key: str = "EMPTY"):
        self.client = OpenAI(base_url=base_url, api_key=api_key)
        self.model = model

    def _clean_output(self, raw_output: str, n: int = 5):
        """
        将模型输出清洗为纯净查询列表
        """
        lines = raw_output.split("\n")
        rewrites = []

        for line in lines:
            line = line.strip()
            if not line:
                continue
            # 去除常见编号、符号前缀
            line = re.sub(r'^[\-\*\d\.、\s]+', '', line)
            # 去除尾部冒号
            line = re.sub(r'[：:\s]+$', '', line)
            # 过滤非查询类提示
            if line and not line.lower().startswith(("以下", "这是", "改写")):
                rewrites.append(line)

        # 去重 + 截断
        rewrites = list(dict.fromkeys(rewrites))[:n]
        return rewrites

    def mmr_select(self, candidates: list, n: int = 5):
        print("MMR 选择，待实现")
        return candidates[:n]
    
    def rewrite(self, question: str, n_variants: int = 5):
        """
        输入一个用户问题，返回多个改写后的查询
        """
        prompt = textwrap.dedent(f"""
            你是一个信息检索专家。给定一个用户问题：
            "{question}"

            请你生成 {n_variants} 个不同的查询，每个查询必须满足以下约束：
            1. 至少包含 1 个 **语义扩展**的改写（multi-query），即保持问题核心但换不同角度/不同表达。
            2. 至少包含 1 个 **子问题拆解**（decomposition），即把复杂问题拆成具体可检索的子问题。
            3. 至少包含 1 个 **对立/反向视角**，保证检索覆盖不同立场。

            要求：
            - 每个改写独立占一行
            - 不要加序号、符号或额外解释
            - 保持自然语言形式
            - 输出示例：
            查询1
            查询2
            查询3
            ...
        """)

        response = self.client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7
        )

        raw_output = response.choices[0].message.content.strip()
        rewrites = self._clean_output(raw_output, n_variants)

        return {
            "original_query": question,
            "rewritten_queries": rewrites
        }

In [4]:
qr = QueryRewriter()

In [5]:
qr._clean_output("""1. 人工智能对就业的未来影响
2. 未来就业趋势与人工智能的关系
3. 人工智能如何改变未来的就业市场
4. 探讨人工智能对未来工作岗位的影响
5. 分析人工智能技术对就业市场的长远影响""")

['人工智能对就业的未来影响',
 '未来就业趋势与人工智能的关系',
 '人工智能如何改变未来的就业市场',
 '探讨人工智能对未来工作岗位的影响',
 '分析人工智能技术对就业市场的长远影响']